<a href="https://colab.research.google.com/github/ritesh23r/LSTM-Next-Word-Predictor/blob/main/LSTM_Next_Word_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("qoute_dataset.csv")

In [ ]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
quotes = df['quote']

# **Preprocessing**

In [ ]:
quotes = quotes.str.lower()

In [ ]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

# **Tokenization**

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
vocab_size = 10000

tokenize = Tokenizer(num_words=vocab_size)
tokenize.fit_on_texts(quotes)

In [ ]:
word_index = tokenize.word_index
print(len(word_index))

8978


In [ ]:
Sequence = tokenize.texts_to_sequences(quotes)

In [ ]:
X=[]
y=[]

for seq in Sequence:
  for i in range (1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [ ]:
max_len = max(len(x) for x in X)

# **padding**

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X , maxlen= max_len, padding='pre')

In [ ]:
y = np.array(y)

# **Vectorization**

In [ ]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [ ]:
y_one_hot.shape

(85271, 10000)

# **Model**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN

In [ ]:
embedding_dim = 30
rnn_units = 128

In [ ]:
rnn_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    SimpleRNN(units=rnn_units),
    Dense(units=vocab_size, activation='softmax')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    LSTM(units=rnn_units),
    Dense(units=vocab_size, activation='softmax')
])

In [ ]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
epochs = 100
batch_size = 128

In [ ]:
history_lstm = lstm_model.fit(X_padded, y_one_hot , epochs=epochs, batch_size=batch_size, validation_split=0.1)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 42s 21ms/step - accuracy: 0.0387 - loss: 6.7627 - val_accuracy: 0.0432 - val_loss: 6.6986
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.0539 - loss: 6.3611 - val_accuracy: 0.0592 - val_loss: 6.6132
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.0722 - loss: 6.1147 - val_accuracy: 0.0808 - val_loss: 6.5001
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.0892 - loss: 5.9003 - val_accuracy: 0.0874 - val_loss: 6.4888
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.0997 - loss: 5.7301 - val_accuracy: 0.0951 - val_loss: 6.4614
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.1086 - loss: 5.5750 - val_accuracy: 0.0955 - val_loss: 6.4646
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.1168 - loss: 5.4231 - val_accuracy: 0.1014 - val_loss: 6.4713
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.1256 - loss: 5

In [ ]:
lstm_model.save('lstm_model.keras')

In [ ]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [ ]:
def predictor(model, tokenizer, text , max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred= model.predict(seq, verbose=0)
  pred_index = np.argmax(pred)

  return index_to_word[pred_index]

In [ ]:
seed_text = "i live in"
next_word = predictor(lstm_model, tokenize, seed_text, max_len)
print(next_word)

love


In [ ]:
def generate_text(model, tokenizer, seed_text, max_len, n_words):
  for _ in range(n_words):
     next_word = predictor(model, tokenizer, seed_text, max_len)
     if next_word == '':
        break
     seed_text += " " + next_word
  return seed_text

In [ ]:
seed_text = 'i love'
generate_text = generate_text(lstm_model, tokenize, seed_text, max_len, 10)
print(generate_text)

i love you like a fat kid loves cake a target the


In [ ]:
import pickle
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenize, handle)


In [ ]:
with open('max_len.pickle', 'wb') as handle:
    pickle.dump(max_len, handle)